### Major refactoring
Since I have so many layers and parameters, I am going to need a better way to organize them. I'm also going to be adding LayerNorm so wayyyy too many parameters in general. 
I'm going to use `nn.Module` class and create my own model.

In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'
words = open('names.txt', 'r').read().splitlines()
device

'cuda'

In [3]:
# encode chars to integers
chars = sorted(list(set(''.join(words))))
stoi = { ch:i+1 for i,ch in enumerate(chars)}
stoi['.'] = 0
itos = { i:ch for ch,i in stoi.items()}
vocab_size = len(itos)

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

In [5]:
# hyperparameters
block_size = 16
batch_size = 32

emb_size = 64
n_hidden = 64

In [6]:
# build dataset
def build_dataset(words):
    X, Y = [], []

    for w in words:
        context = [0] * block_size
        # sliding window
        for ch in w + '.':
            ix = stoi[ch] # encode(ch)
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

n1 = int(0.9*len(words))

X, Y = build_dataset(words)
Xtr, Ytr = build_dataset(words[:n1])
Xval, Yval = build_dataset(words[n1:])

def get_batch(split):
    dataX = Xtr if split == 'train' else Xval
    dataY = Ytr if split == 'train' else Yval

    ix = torch.randint(low=0, high=dataX.shape[0], size=(batch_size, ))
    x = torch.stack([dataX[i] for i in ix])
    y = torch.stack([dataY[i] for i in ix])

    x, y = x.to(device), y.to(device)
    return x, y


print(X[:5])
print(Y[:5])
len(Xtr)

tensor([[ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  5],
        [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  5, 13],
        [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  5, 13, 13],
        [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  5, 13, 13,  1]])
tensor([ 5, 13, 13,  1,  0])


205411

For fun, I will recreate some boilerplate layers.

In [ ]:
class Linear(nn.Module):
  def __init__(self, fan_in, fan_out, bias=True):
    self.weight = torch.randn((fan_in, fan_out)) / fan_in**0.5 # note: kaiming init
    self.bias = torch.zeros(fan_out) if bias else None
  
  def __forward__(self, x):
    self.out = x @ self.weight
    if self.bias is not None:
      self.out += self.bias
    return self.out

In [ ]:
class Flatten(nn.Module):
    def __init__(self):

    def __forward__(self):
        

In [ ]:
class DilatedCasualConvolutions():
    def __init__(self):

    def __forward__(self):
        

In [ ]:
class WaveNet(nn.Module):

    def __init__(self):
        super().__init__()

        self.tok_emb = nn.Embedding(vocab_size, emb_size)
        self.pos_emb = nn.Embedding(block_size, emb_size)
        
        self.unembed = Linear(n_hidden, vocab_size)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx):

    def generate(self, idx):
